Import packages 

In [15]:
import os
import re
from dotenv import load_dotenv

# RAG import packages 
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
# Vector Embeddings
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


Parse txt files into column-level chunks

In [16]:
def parse_schema_to_column_chunks(docs_dir: str) -> list[Document]:
    """Parse each COLUMN block into a separate Document with table metadata."""
    chunks = []
    
    for filename in os.listdir(docs_dir):
        if not filename.endswith(".txt"):
            continue
            
        filepath = os.path.join(docs_dir, filename)
        table_name = filename.replace(".txt", "")
        
        with open(filepath, "r") as f:
            content = f.read()
        
        # Extract TABLE line for context
        table_match = re.search(r"TABLE: (\w+) \((\d+) rows\)", content)
        joins_match = re.search(r"JOINS: (.+)", content)
        joins = joins_match.group(1) if joins_match else ""
        
        # Split by COLUMN blocks
        column_blocks = re.split(r"\n(?=COLUMN(?: NAME)?: )", content)
        
        for block in column_blocks:
            if not block.strip() or block.startswith("TABLE:") or block.startswith("JOINS:"):
                continue
            col_match = re.search(r"COLUMN NAME: (.+?) \(Use this when querying information\)", block)
            if not col_match:
                col_match = re.search(r"COLUMN: (\w+(?:\s+\w+)?)", block)
            if not col_match:
                continue
            col_name = col_match.group(1).strip()
            
            # Full column block as chunk content (Data Element, Description, etc.)
            chunk_content = f"TABLE: {table_name}\n{block.strip()}"
            if joins:
                chunk_content += f"\n\nJOINS: {joins}"
            
            doc = Document(
                page_content=chunk_content,
                metadata={
                    "table": table_name,
                    "column": col_name,
                    "source": filename
                }
            )
            chunks.append(doc)
    
    return chunks

Load into vector store and run retrieval

In [17]:
from pathlib import Path

# Same persist dir as retrieval/graph/tool/vectorRag.py: retrieval/chroma_db
_base = Path.cwd()
if _base.name != "retrieval" and (_base / "retrieval").is_dir():
    _base = _base / "retrieval"
CHROMA_DIR = (_base / "chroma_db").resolve()
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# Parse to column-level chunks
docs_dir = "documents_for_vectordb"
chunks = parse_schema_to_column_chunks(docs_dir)

# OpenRouter Embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# Then use embeddings in Chroma
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="schema_columns",
    persist_directory=str(CHROMA_DIR),
)

# Retrieve relevant columns for a user query
query = "How many patients died and what was the cause?"
retrieved = vectorstore.similarity_search(query, k=5)

for doc in retrieved:
    print(f"Table: {doc.metadata['table']}, Column: {doc.metadata['column']}")
    print(doc.page_content[:200], "...\n")

Table: death, Column: person_id
TABLE: death
COLUMN NAME: person_id (Use this when querying information)
  Description: Deidentified patient identifier. Links to person.
  Datatype: integer
  Key: FK to person

JOINS: Join to person ...

Table: death, Column: cause_source_value
TABLE: death
COLUMN NAME: cause_source_value (Use this when querying information)
  Description: Source code for cause of death.
  Datatype: varchar(50)
  Key: —

JOINS: Join to person via death.perso ...

Table: death, Column: death_date
TABLE: death
COLUMN NAME: death_date (Use this when querying information)
  Description: Date of death.
  Datatype: date
  Key: —

JOINS: Join to person via death.person_id = person.person_id. Link wi ...

Table: person, Column: year_of_birth
TABLE: person
COLUMN NAME: year_of_birth (Use this when querying information)
  Description: Year of birth of patient.
  Datatype: integer
  Key: —

JOINS: Person is the central table. All other table ...

Table: condition_occurrence, Colu

Parse cancer.txt and sql_template.txt, create vector databases

Build Chroma collections:
- **icd10_cancer_reference**: ICD-10-AM cancer codes and descriptions from cancer.txt (for mapping cancer types to codes)
- **sql_templates**: Few-shot SQL examples from sql_template.txt (for retrieving relevant query patterns)

In [18]:
from pathlib import Path

# Paths (relative to retrieval/)
CANCER_PATH = "icd-am/cancer.txt"
SQL_TEMPLATE_PATH = "sql_template/sql_template.txt"
_base = Path.cwd()
if _base.name != "retrieval" and (_base / "retrieval").is_dir():
    _base = _base / "retrieval"
CHROMA_DIR = str((_base / "chroma_db").resolve())
Path(CHROMA_DIR).mkdir(parents=True, exist_ok=True)

In [19]:
def parse_cancer_txt(path: str) -> list[Document]:
    """Parse cancer.txt into Documents by section. Sections are separated by ==== lines; title and content alternate."""
    with open(path, "r") as f:
        content = f.read()
    parts = re.split(r"\n=+\n", content)
    docs = []
    # parts[0] = intro; parts[1]=title, parts[2]=content, parts[3]=title, parts[4]=content, ...
    for i in range(1, len(parts) - 1, 2):
        title = parts[i].strip()
        content_block = parts[i + 1].strip() if i + 1 < len(parts) else ""
        if not title or not content_block:
            continue
        combined = f"{title}\n\n{content_block}"
        doc = Document(
            page_content=combined,
            metadata={"source": "cancer.txt", "type": "icd10_cancer_reference"},
        )
        docs.append(doc)
    return docs

In [20]:
# Parse and build cancer vector DB
cancer_docs = parse_cancer_txt(CANCER_PATH)
print(f"Parsed {len(cancer_docs)} cancer reference sections")

cancer_vectorstore = Chroma.from_documents(
    documents=cancer_docs,
    embedding=embeddings,
    collection_name="icd10_cancer_reference",
    persist_directory=CHROMA_DIR,
)
print("Created chroma collection: icd10_cancer_reference")


Parsed 12 cancer reference sections
Created chroma collection: icd10_cancer_reference


Testing the cancer db

In [21]:
print("\nTest: cancer query 'colorectal' ->", cancer_vectorstore.similarity_search("colorectal cancer", k=1)[0].page_content[:150], "...")


Test: cancer query 'colorectal' -> COLORECTAL CANCER (C18–C21)

C18  Colon cancer
  C18.0  Malignant neoplasm of caecum
  C18.1  Malignant neoplasm of appendix
  C18.2  Malignant neopla ...


Parsing sql template documents

In [22]:
def parse_sql_template_txt(path: str) -> list[Document]:
    """Parse sql_template.txt into Documents by block (split by ---)."""
    with open(path, "r") as f:
        content = f.read()
    # Skip header comment lines
    blocks = content.split("\n---\n")
    docs = []
    for block in blocks:
        block = block.strip()
        if not block or block.startswith("#"):
            continue
        doc = Document(
            page_content=block,
            metadata={"source": "sql_template.txt", "type": "sql_template"},
        )
        docs.append(doc)
    return docs

In [23]:
# Parse and build SQL template vector DB
sql_docs = parse_sql_template_txt(SQL_TEMPLATE_PATH)
print(f"Parsed {len(sql_docs)} SQL template blocks")

sql_template_vectorstore = Chroma.from_documents(
    documents=sql_docs,
    embedding=embeddings,
    collection_name="sql_templates",
    persist_directory=CHROMA_DIR,
)
print("Created chroma collection: sql_templates")



Parsed 20 SQL template blocks
Created chroma collection: sql_templates


Test for the sql template db

In [24]:
print("\nTest: sql query 'patients 2020' ->", sql_template_vectorstore.similarity_search("total patients diagnosed 2020", k=1)[0].page_content[:200], "...")


Test: sql query 'patients 2020' -> HEADER: How many different types of diagnoses have been recorded under the ICD10 classification?
TIP: count, distinct
SQL:
SELECT COUNT(DISTINCT ICD10) AS count FROM condition_occurrence; ...
